# Restore Solar Filament Segmentation Challenge 2026 dataset to NEW Google Drive

This notebook will download the **official competition data from Kaggle**, verify that training images, test images and annotations are present, save an archive in the target Drive folder, and split it into files smaller than ChatGPT's Drive connector limit.

The destination folder was created on the **new Google Drive account**; its ID is stored in this notebook. It will never write into the old account. The original ZIP and parts together need approximately **1.4 GB of free Drive space**.

Run **Runtime → Run all** in Google Colab; approve Google authorization for the new account. If Kaggle requests a token, use **Kaggle → Settings → API → Generate New Token** and provide it only through the interactive login prompt or Colab secret. You must have accepted the [competition rules](https://www.kaggle.com/competitions/filament-segmentation-2026/rules). Do not paste tokens into the source or ChatGPT.


In [ ]:
%pip -q install -U kagglehub
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
creds, _ = google.auth.default()
drive_api = build('drive','v3',credentials=creds,cache_discovery=False)
FOLDER_ID = "1aA5QZgdnJVJdiudUAJWh2yoQN65oNmka"
target = drive_api.files().get(fileId=FOLDER_ID,fields='id,name,mimeType').execute()
assert target['mimeType']=='application/vnd.google-apps.folder', 'The Drive destination is not a folder.'
print('Verified the new Drive folder:',target['name'],target['id'])


In [ ]:
import os
from pathlib import Path
import kagglehub
from google.colab import userdata
try:
    token=userdata.get('KAGGLE_API_TOKEN')
except Exception:
    token=None
if token: os.environ['KAGGLE_API_TOKEN']=token
work=Path('/content/magfilo_kaggle_download')
work.mkdir(exist_ok=True)
try:
    downloaded=Path(kagglehub.competition_download('filament-segmentation-2026',output_dir=str(work)))
except Exception as exc:
    if not any(s in str(exc).lower() for s in ('401','403','unauthorized','forbidden','credential','token','permission','authentication')):
        raise
    print('Kaggle authentication required. Log in with the Kaggle account that accepted the rules.')
    kagglehub.login()
    downloaded=Path(kagglehub.competition_download('filament-segmentation-2026',output_dir=str(work),force_download=True))
print('Downloaded:',downloaded)


In [ ]:
import zipfile, shutil
def valid_zip(p):
    try:
        with zipfile.ZipFile(p) as z:
            names=[x.lower().replace('\\','/') for x in z.namelist()]
        return (any('train_images/' in n and n.endswith(('.jpg','.jpeg')) for n in names)
                and any('test_images/' in n and n.endswith(('.jpg','.jpeg')) for n in names)
                and any('annotations' in n and n.endswith('.json') for n in names))
    except (OSError,zipfile.BadZipFile):
        return False

candidates=[downloaded] if downloaded.is_file() else list(downloaded.rglob('*'))
source_zip=next((p for p in candidates if p.is_file() and p.suffix.lower()=='.zip' and valid_zip(p)),None)
archive=Path('/content/filament-segmentation-2026.zip')
if source_zip:
    if source_zip.resolve()!=archive.resolve(): shutil.copyfile(source_zip,archive)
else:
    files=[p for p in candidates if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.json')]
    if not files: raise RuntimeError('No images or JSON were downloaded.')
    base=downloaded if downloaded.is_dir() else downloaded.parent
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
        for p in sorted(files): z.write(p,p.relative_to(base).as_posix())
if not valid_zip(archive): raise RuntimeError('Train/test images or annotations missing; upload aborted.')
with zipfile.ZipFile(archive) as z:
    print('Verified',sum(n.endswith(('.jpg','.jpeg')) for n in z.namelist()),'images and',sum(n.endswith('.json') for n in z.namelist()),'JSON files.')


In [ ]:
import hashlib,math,json
from googleapiclient.http import MediaFileUpload
PART_SIZE=180*1024*1024
BUFFER=8*1024*1024
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as src:
        for data in iter(lambda:src.read(BUFFER),b''): h.update(data)
    return h.hexdigest()
files_to_upload=[archive]
parts=[]
with archive.open('rb') as source:
    for idx in range(1,math.ceil(archive.stat().st_size/PART_SIZE)+1):
        path=Path('/content') / (archive.name+f'.part{idx:02d}')
        remaining=min(PART_SIZE,archive.stat().st_size-(idx-1)*PART_SIZE)
        with path.open('wb') as out:
            while remaining:
                chunk=source.read(min(BUFFER,remaining))
                if not chunk: raise IOError('Unexpected end of ZIP')
                out.write(chunk);remaining-=len(chunk)
        parts.append({'name':path.name,'bytes':path.stat().st_size,'sha256':sha256(path)})
        files_to_upload.append(path)
manifest={'archive':archive.name,'bytes':archive.stat().st_size,'sha256':sha256(archive),'parts':parts}
manifest_file=Path('/content/filament-segmentation-2026.parts.json')
manifest_file.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
files_to_upload.append(manifest_file)
for path in files_to_upload:
    safe=path.name.replace("'","\\'")
    existing=drive_api.files().list(q=f"'{FOLDER_ID}' in parents and name = '{safe}' and trashed = false",
        fields='files(id,name,size,md5Checksum)',page_size=100).execute().get('files',[])
    if existing:
        # Preserve user data: no overwrite, and do not silently assume matching size means matching bytes.
        raise FileExistsError(f'Already exists in Drive, not overwriting: {path.name}')
    media=MediaFileUpload(str(path),mimetype='application/zip' if path==archive else 'application/octet-stream',
        chunksize=8*1024*1024,resumable=True)
    request=drive_api.files().create(body={'name':path.name,'parents':[FOLDER_ID]},media_body=media,
        fields='id,name,size,md5Checksum')
    result=None
    while result is None:
        status,result=request.next_chunk()
    print('Saved:',result['name'],result.get('size',''), 'bytes')
print('FINISHED: data archive, chunks and integrity manifest are now in the new Drive account.')
